# 教會資源小樣本爬蟲（CPU，不需 GPU）

從指定的恢復本聖經與生命讀經頁面追蹤站內連結，最多10頁／20MiB／180秒。保留原始HTML、正文、來源時間、SHA-256及音訊網址。也保存Common Voice官方資料目錄文件。

**本次只收集文字與音訊索引，不下載完整音訊、不訓練。** TWGBR授權尚未取得；文字放在待審區，不能把網頁文章當作已確認逐字稿。Common Voice目錄不是已下載語料。

依序執行下方4格，允許掛載Drive。若網站拒絕或robots無法檢查，保留報告並停止，不反覆重跑。已完成頁面會核對hash後續用。


In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path('/content/Meeting-Live-Captions-crawler')
REMOTE = 'https://github.com/timo893131/Meeting-Live-Captions.git'
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1',REMOTE,str(REPO)],check=True)
else:
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=REPO,text=True).strip()
    if origin != REMOTE or subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
        raise RuntimeError('既有工作目錄來源不符或有改動，請保留原檔並使用新的目錄。')
    subprocess.run(['git','fetch','--depth','1','origin','main'],cwd=REPO,check=True)
    subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO,check=True)
print('程式版本：',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','beautifulsoup4==4.14.3','soupsieve==3.0.3','typing_extensions==4.16.0'],check=True)
print('爬蟲已就緒；不需要GPU、Whisper或Hugging Face帳號。')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = Path('/content/drive/MyDrive/meeting-live-captions/source-collection/twgbr-v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('資料保存位置：',OUTPUT)


In [ ]:
command = [sys.executable,'-u',str(REPO/'scripts/crawl_church_sources.py'),'--output',str(OUTPUT)]
process = subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
for line in process.stdout:
    print(line,end='',flush=True)
code = process.wait()
print('爬蟲退出狀態：',code)
if code:
    print('來源擷取未完成；請保留報告，不要連續重試。沒有開始訓練。')


In [ ]:
import json
from google.colab import files
REPORT = OUTPUT/'reports/collection-report.json'
if REPORT.exists():
    report = json.loads(REPORT.read_text())
    print(json.dumps({key:report.get(key) for key in ['status','pages_acquired','text_documents','audio_urls_discovered','catalog_documents','audio_downloaded','usable_paired_hours','failures']},ensure_ascii=False,indent=2))
    print('完整資料留在Drive；只下載報告供檢查。')
    files.download(str(REPORT))
else:
    print('沒有完成報告，請查看上一格的錯誤輸出。')
